In [1]:
pip install pandas numpy yfinance statsmodels scikit-learn matplotlib seaborn TA-Lib tqdm

Note: you may need to restart the kernel to use updated packages.


In [1]:
"""
Код должен построить три модели классического машинного обучения (ML) для формирования торговых стратегий на ежедневных данных цен Close (только close и data) по акциям и криптовалютам (stocks=['MSTR', 'BRPHF'], cryptos=['BTC-USD', 'SOL-USD']), которые загружаются с Yahoo Finance, начиная c 2020-01-01. 
Для каждого инструмента код должен:
1. Обработать загруженные ежедневные данные (только close и data), т.е. выполнить очистку данных, удаление выбросов, заполнение пропусков и устранение дубликатов, и далее нормализацию обработанных выше данных (методом Z-Score). Кроме того, код должен обеспечить пополнение (обновление) данных. Поскольку исходные данные представляют собой временные ряды, требуется применить необходимые методы, которые позволяют эффективно бороться с переобучением модели ML. Сохранить данные в csv-файл. Отображать процессы обработки, нормализации и сохранения. 
2. Дополнить торговую стратегию Strategy1 (см ниже) на основе индикаторов EMA+RSI+MACD тремя моделями классического машинного обучения и сформировать три торговые стратегии с разными моделями ML ((gradient_boosting, random_forest, logistic_regression)), с целью прогнозирования оптимального торгового решения (покупку или продажу, открытия  или закрытия позиции). Требуется использовать продвинутые алгоритмы ML на основе градиентного бустинга, а также проводить оптимизацию параметров моделей для стабилизации модели и повышения точности предсказаний. Требуется отобразить процесс оптимизацию и параметры. Отобразить определения трех торговых стратегий с ML с параметрами и предоставить описания этих моделей с параметрами для дальнейшего сравнения в дашборде.
3. Разделить нормализованные обработанные данные на train и valid наборы данных. Провести бэктестирование торговых стратегий с моделями ML на train наборе данных, оптимизировать параметры. Провести бэктестирование на valid  наборе данных с оптимальными гиперпараметрами, полученными на train наборе. Предоставить метрики для дальнейшего сравнения в дашборде. Выполнить оценку моделей ML, использованных в торговых стратегиях, с регуляризацией и с оптимизацией параметров. В отдельной таблице отображать классификационные метрики моделей ML: Cutoff, Precision, , Recall, Accuracy, F1-Score, ROC AUC. Требуется использовать методы регуляризации: Lasso (Least Absolute Shrinkage and Selection Operator), L2 регуляризация (Ridge регуляризация), Elastic Net. Для деревьев решений требуется использовать методы: max_depth, min_samples_split, min_samples_leaf, max_leaf_nodes, Cost Complexity Pruning, max_features, bootstrap. Требуется отобразить используемые методы регуляризации и классификационные метрики моделей для дальнейшего сравнения в дашборде.
4. Провести бэктестирование торговых стратегий без моделей ML на train наборе данных, оптимизировать гиперпараметры, провести бэктестирование на valid наборе данных с оптимальными гиперпараметрами, полученными на train наборе. Предоставить метрики для дальнейшего сравнения в дашборде. 
В итоге код должен сформировать дашборд с комбо-боксом для выбора инструмента над сводной таблицей с метриками протестированных стратегий на разных наборах данных (train, valid), показывающие эффективность трех торговых стратегий с моделями ML и одной без моделей ML. В сводной таблице дашборд должен отображать строки со значениями метрик производительности для следующих столбцов: для бэктестирования без ML, для бэктестировании с моделью gradient_boosting, для бэктестировании с моделью random_forest, для бэктестировании с моделью logistic_regression. Под сводной таблицей с метриками должна отображаться таблица с графиками, по четыре графика в одном ряду: график с инструментом, график Profit/Loss, график win rate, график maximal dropdawn. В каждой строке таблицы с  графиками должны отображаться графики для стратегий без ML, стратегии с gradient_boosting, стратегии с random_forest и стратегии с logistic_regression.

Copyright (c) 2025 Matvei Vasetsov (Матвей Васецов). All rights reserved.
Этот код является интеллектуальной собственностью и защищен авторским правом. Любое использование, копирование или распространение без разрешения автора запрещено.

"""

import pandas as pd
import numpy as np
from datetime import datetime
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, mean_squared_error, r2_score, roc_curve, auc, roc_auc_score
from sklearn.linear_model import Lasso, Ridge, ElasticNet
import matplotlib.pyplot as plt
import seaborn as sns
import talib as ta
import logging
from pathlib import Path
import streamlit as st
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from tqdm import tqdm
from sklearn.tree import DecisionTreeClassifier
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class DataHandler:
    def __init__(self):
        self.scaler = StandardScaler()
        
    def download_data(self, symbols, start_date):
        """Download and clean price data from Yahoo Finance"""
        data_dict = {}
        
        # Ensure symbols is a list
        if isinstance(symbols, str):
            symbols = [symbols]
            
        for symbol in symbols:
            try:
                # Validate symbol format
                symbol = str(symbol).strip().upper()
                if not symbol:
                    continue
                    
                logging.info(f"Attempting to download data for {symbol}")
                
                # Create Ticker object first
                ticker = yf.Ticker(symbol)
                
                # Download data using the history method
                data = ticker.history(
                    start=start_date,
                    end=datetime.now().strftime('%Y-%m-%d'),
                    interval='1d',
                    auto_adjust=True
                )
                
                if data is None or data.empty:
                    logging.warning(f"No data retrieved for {symbol}")
                    continue
                
                # Extract only Close prices and ensure proper structure
                if 'Close' in data.columns:
                    data = data[['Close']].copy()
                    data.index = pd.to_datetime(data.index)
                    data = data.sort_index()
                    
                    # Remove any potential string/text values and invalid prices
                    data['Close'] = pd.to_numeric(data['Close'], errors='coerce')
                    data = data[data['Close'] > 0]  # Remove zero or negative prices
                    data = data.dropna()
                    
                    if not data.empty:
                        data_dict[symbol] = data
                        logging.info(f"Successfully downloaded data for {symbol}: {data.shape[0]} records")
                        
                        # Save clean data to CSV
                        self._save_to_csv(symbol, data)
                    else:
                        logging.warning(f"No valid price data for {symbol} after cleaning")
                else:
                    logging.warning(f"No Close price column found for {symbol}")
                    
            except Exception as e:
                logging.error(f"Error downloading {symbol}: {str(e)}")
                # Try alternative download method if first method fails
                try:
                    logging.info(f"Attempting alternative download method for {symbol}")
                    data = pd.DataFrame(yf.download(
                        symbol,
                        start=start_date,
                        end=datetime.now().strftime('%Y-%m-%d'),
                        progress=False
                    ))
                    
                    if not data.empty and 'Close' in data.columns:
                        data = data[['Close']].copy()
                        data.index = pd.to_datetime(data.index)
                        data = data.sort_index()
                        data['Close'] = pd.to_numeric(data['Close'], errors='coerce')
                        data = data[data['Close'] > 0]
                        data = data.dropna()
                        
                        if not data.empty:
                            data_dict[symbol] = data
                            logging.info(f"Successfully downloaded data using alternative method for {symbol}: {data.shape[0]} records")
                            self._save_to_csv(symbol, data)
                        else:
                            logging.warning(f"No valid price data for {symbol} after cleaning (alternative method)")
                except Exception as e2:
                    logging.error(f"Alternative download method also failed for {symbol}: {str(e2)}")
                continue
                
        if not data_dict:
            logging.warning("No data was downloaded for any symbol")
            
        return data_dict
        
    def _save_to_csv(self, symbol, data):
        """Helper method to save data to CSV"""
        try:
            output_dir = Path('data/raw')
            output_dir.mkdir(parents=True, exist_ok=True)
            
            output_file = output_dir / f"{symbol.replace('/', '_')}_raw.csv"
            
            # Save only the clean numeric data with proper date index
            data.to_csv(
                output_file,
                header=True,
                index=True,
                index_label='Date',
                float_format='%.2f'  # Format numbers to 2 decimal places
            )
            logging.info(f"Saved clean raw data for {symbol} to {output_file}")
            
        except Exception as e:
            logging.error(f"Error saving data for {symbol}: {str(e)}")

    def __init__(self):
        self.scaler = StandardScaler()

    def preprocess_data(self, data):
        """Clean and preprocess the data with progress bar"""
        try:
            if data is None or (isinstance(data, pd.DataFrame) and data.empty):
                raise ValueError("Empty or None data provided")
                
            df = data.copy()
            
            # Create progress bar
            pbar = tqdm(total=5, desc="Preprocessing Data")
            
            # Ensure we have a DataFrame
            if isinstance(df, pd.Series):
                df = df.to_frame(name='Close')
            pbar.update(1)
            
            # Convert to numeric and remove any non-numeric values
            df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
            df = df.dropna()
            pbar.update(1)
            
            # Remove duplicates
            df = df[~df.index.duplicated(keep='first')]
            pbar.update(1)
            
            # Remove outliers using IQR method
            if len(df) > 10:
                Q1 = df['Close'].quantile(0.25)
                Q3 = df['Close'].quantile(0.75)
                IQR = Q3 - Q1
                filter_mask = (df['Close'] >= (Q1 - 3.0 * IQR)) & (df['Close'] <= (Q3 + 3.0 * IQR))
                df = df[filter_mask]
            pbar.update(1)
            
            # Final validation
            if df.empty:
                raise ValueError("No valid numeric data after preprocessing")
            
            if not np.issubdtype(df['Close'].dtype, np.number):
                raise ValueError("Close prices are not numeric")
            
            pbar.update(1)
            pbar.close()
                
            logging.info(f"Successfully preprocessed data: {df.shape[0]} records")
            return df
            
        except Exception as e:
            logging.error(f"Error in preprocess_data: {str(e)}")
            return pd.DataFrame()

    def normalize_data(self, data):
        """Normalize data with progress bar"""
        try:
            if data.empty:
                return data
            
            # Create progress bar
            pbar = tqdm(total=3, desc="Normalizing Data")
            
            # Ensure we have numeric data
            data = data.copy()
            data['Close'] = pd.to_numeric(data['Close'], errors='coerce')
            data = data.dropna()
            pbar.update(1)
            
            if data.empty:
                raise ValueError("No valid numeric data to normalize")
            
            # Apply normalization
            normalized_data = pd.DataFrame(
                self.scaler.fit_transform(data),
                columns=data.columns,
                index=data.index
            )
            pbar.update(1)
            
            logging.info(f"Normalized data shape: {normalized_data.shape}")
            
            pbar.update(1)
            pbar.close()
            
            return normalized_data
            
        except Exception as e:
            logging.error(f"Error in normalize_data: {str(e)}")
            return pd.DataFrame()

    def prepare_ml_features(self, data):
        """Prepare features for ML models with explicit dimension handling"""
        try:
            if data.empty:
                raise ValueError("Empty data provided")
            
            df = data.copy()
            
            if isinstance(df, pd.Series):
                df = df.to_frame()
                df.columns = ['Close']
            
            df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
            close_prices = df['Close'].values
            
            if len(close_prices) < 30:
                raise ValueError(f"Insufficient data points: {len(close_prices)}")
            
            # Calculate technical indicators
            try:
                # Расчет индикаторов с правильными параметрами
                ema20 = ta.EMA(close_prices, timeperiod=20)
                rsi = ta.RSI(close_prices, timeperiod=14)
                macd, signal, _ = ta.MACD(
                    close_prices, 
                    fastperiod=12, 
                    slowperiod=26, 
                    signalperiod=9
                )
                
                # Создаем колонки с заполнением пропусков
                df['EMA20'] = pd.Series(ema20, index=df.index).ffill().bfill()
                df['RSI'] = pd.Series(rsi, index=df.index).ffill().bfill()
                df['MACD'] = pd.Series(macd, index=df.index).ffill().bfill()
                df['Signal'] = pd.Series(signal, index=df.index).ffill().bfill()
                
            except Exception as e:
                logging.error(f"Error calculating indicators: {str(e)}")
                # Создаем базовые колонки при ошибке расчета
                df['EMA20'] = df['Close'].rolling(20).mean()
                df['RSI'] = 50.0
                df['MACD'] = 0.0
                df['Signal'] = 0.0
            
            # Остальная часть метода без изменений
            df['MACD_Hist'] = df['MACD'] - df['Signal']
            df['Returns'] = df['Close'].pct_change()
            df['Returns'] = df['Returns'].replace([np.inf, -np.inf], np.nan)
            
            for i in range(1, 6):
                df[f'Returns_Lag{i}'] = df['Returns'].shift(i)
                df[f'Price_Lag{i}'] = df['Close'].shift(i)
            
            df['Volatility'] = df['Returns'].rolling(window=20, min_periods=1).std()
            df['MA20'] = df['Close'].rolling(window=20, min_periods=1).mean()
            df['MA_Crossover'] = df['Close'] - df['MA20']
            df['RSI_Change'] = df['RSI'].diff()
            
            df['Target'] = np.where(df['Close'].shift(-1) > df['Close'], 1, -1)
            df = df.replace([np.inf, -np.inf], np.nan)
            df = df.ffill().bfill().dropna()
            
            logging.info(f"Prepared features shape: {df.shape}")
            return df
            
        except Exception as e:
            logging.error(f"Error in prepare_ml_features: {str(e)}")
            return pd.DataFrame()

    def split_data(self, data, train_ratio=0.8):
        """Split data with time-aware indexing"""
        dates = data.index.sort_values()
        train_size = int(len(dates) * train_ratio)
        train_dates = dates[:train_size]
        valid_dates = dates[train_size:]
        return data.loc[train_dates].copy(), data.loc[valid_dates].copy()

class MLModelHandler:
    def __init__(self):
        self.models = {
            'gradient_boosting': GradientBoostingClassifier(random_state=42),
            'random_forest': RandomForestClassifier(random_state=42),
            'logistic_regression': LogisticRegression(random_state=42)
        }
        self.regularization_results = {}
        
        # Initialize regularization models with proper parameters
        self.regularization_models = {
            'lasso': Lasso(alpha=0.01, max_iter=10000),
            'ridge': Ridge(alpha=0.01, max_iter=10000),
            'elastic_net': ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000)
        }
        
        self.param_grids = {
            'gradient_boosting': {
                'n_estimators': [100, 200],
                'learning_rate': [0.01, 0.1],
                'max_depth': [3, 5],
                'min_samples_split': [2, 5],
                'subsample': [0.8, 1.0]
            },
            'random_forest': {
                'n_estimators': [100, 200],
                'max_depth': [3, 5],
                'min_samples_split': [2, 5],
                'max_features': ['sqrt', 'log2'],
                'bootstrap': [True],
                'ccp_alpha': [0.0, 0.1]
            },
            'logistic_regression': {
                'C': [0.1, 1.0, 10.0],
                'penalty': ['l1', 'l2', 'elasticnet'],
                'solver': ['saga'],
                'l1_ratio': [0.5]
            }
        }

    def train_models(self, X_train, y_train):
        """Train ML models with improved handling"""
        trained_models = {}
        
        try:
            X_train = np.asarray(X_train)
            y_train = np.asarray(y_train)
            
            if len(X_train.shape) == 1:
                X_train = X_train.reshape(-1, 1)
            
            if len(y_train.shape) == 2:
                y_train = y_train.ravel()
            
            # Updated parameter grids for better performance
            param_grids = {
                'gradient_boosting': {
                    'n_estimators': [100, 200],
                    'learning_rate': [0.01, 0.05],
                    'max_depth': [3, 4],
                    'min_samples_split': [5, 10],
                    'subsample': [0.8, 0.9]
                },
                'random_forest': {
                    'n_estimators': [100, 200],
                    'max_depth': [3, 4],
                    'min_samples_split': [5, 10],
                    'min_samples_leaf': [2, 4],
                    'max_features': ['sqrt']
                },
                'logistic_regression': {
                    'C': [0.1, 1.0],
                    'class_weight': ['balanced'],
                    'max_iter': [1000],
                    'solver': ['lbfgs']
                }
            }
            
            tscv = TimeSeriesSplit(n_splits=3)
            
            for name, model in self.models.items():
                try:
                    grid_search = GridSearchCV(
                        model,
                        param_grids[name],
                        cv=tscv,
                        scoring='f1',
                        n_jobs=-1
                    )
                    
                    grid_search.fit(X_train, y_train)
                    trained_models[name] = grid_search.best_estimator_
                    
                except Exception as e:
                    logging.error(f"Error training {name} model: {str(e)}")
                    continue
                    
            return trained_models
            
        except Exception as e:
            logging.error(f"Error in train_models: {str(e)}")
            return {}

    def plot_optimization_results(grid_search):
        results = grid_search.cv_results_
        plt.figure(figsize=(10, 6))
        plt.plot(results['param_n_estimators'], results['mean_test_score'], marker='o')
        plt.xlabel('n_estimators')
        plt.ylabel('Mean Test Score')
        plt.title('Grid Search Results')
        plt.show()

    def train_regularization_models(self, X_train, y_train):
        """Train regularization models and visualize results"""
        results = {}
        alphas = [0.0001, 0.001, 0.01, 0.1, 1, 10]
        
        for name, model_class in [
            ('Lasso', Lasso), 
            ('Ridge', Ridge), 
            ('ElasticNet', ElasticNet)
        ]:
            mse_scores = []
            r2_scores = []
            coef_paths = []
            
            for alpha in tqdm(alphas, desc=f"Training {name}"):
                if name == 'ElasticNet':
                    model = model_class(alpha=alpha, l1_ratio=0.5, max_iter=10000)
                else:
                    model = model_class(alpha=alpha, max_iter=10000)
                
                model.fit(X_train, y_train)
                y_pred = model.predict(X_train)
                
                mse_scores.append(mean_squared_error(y_train, y_pred))
                r2_scores.append(r2_score(y_train, y_pred))
                coef_paths.append(model.coef_)
            
            results[name] = {
                'alphas': alphas,
                'mse_scores': mse_scores,
                'r2_scores': r2_scores,
                'coef_paths': coef_paths
            }
        
        # Plot regularization results
        self._plot_regularization_results(results)
        
        return results

    def _plot_regularization_results(self, results):
        """Visualize regularization results"""
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        fig.suptitle('Regularization Analysis')
        
        for idx, (name, result) in enumerate(results.items()):
            # Plot MSE scores
            axes[0, idx].plot(result['alphas'], result['mse_scores'], marker='o')
            axes[0, idx].set_xscale('log')
            axes[0, idx].set_xlabel('Alpha')
            axes[0, idx].set_ylabel('MSE')
            axes[0, idx].set_title(f'{name} - MSE vs Alpha')
            
            # Plot coefficient paths
            coef_paths = np.array(result['coef_paths'])
            axes[1, idx].plot(result['alphas'], coef_paths)
            axes[1, idx].set_xscale('log')
            axes[1, idx].set_xlabel('Alpha')
            axes[1, idx].set_ylabel('Coefficients')
            axes[1, idx].set_title(f'{name} - Coefficient Paths')
        
        plt.tight_layout()
        plt.show()

    def evaluate_models(self, X_train, y_train, X_test, y_test, models):
        """Evaluate ML models with ROC AUC"""
        results = {}
        for name, model in models.items():
            try:
                # Для классификаторов с вероятностями
                if hasattr(model, "predict_proba"):
                    y_train_proba = model.predict_proba(X_train)[:, 1]
                    y_test_proba = model.predict_proba(X_test)[:, 1]
                    train_roc_auc = roc_auc_score(y_train, y_train_proba)
                    valid_roc_auc = roc_auc_score(y_test, y_test_proba)
                else:
                    train_roc_auc = None
                    valid_roc_auc = None

                # Остальные метрики
                y_train_pred = model.predict(X_train)
                train_report = classification_report(y_train, y_train_pred, output_dict=True)
                
                y_test_pred = model.predict(X_test)
                test_report = classification_report(y_test, y_test_pred, output_dict=True)
                
                results[name] = {
                    'train': {
                        'roc_auc': train_roc_auc,
                        'accuracy': accuracy_score(y_train, y_train_pred),
                        'precision': precision_score(y_train, y_train_pred),
                        'recall': recall_score(y_train, y_train_pred),
                        'f1': f1_score(y_train, y_train_pred),
                        'report': train_report
                    },
                    'valid': {
                        'roc_auc': valid_roc_auc,
                        'accuracy': accuracy_score(y_test, y_test_pred),
                        'precision': precision_score(y_test, y_test_pred),
                        'recall': recall_score(y_test, y_test_pred),
                        'f1': f1_score(y_test, y_test_pred),
                        'report': test_report
                    }
                }
            except Exception as e:
                logging.error(f"Error evaluating {name} model: {str(e)}")
        return results

    def visualize_feature_importance(self, model_name, feature_names):
        """Визуализация важности признаков для моделей ML"""
        if hasattr(self.models[model_name], 'feature_importances_'):
            importances = self.models[model_name].feature_importances_
            indices = np.argsort(importances)[::-1]
            
            plt.figure(figsize=(10, 6))
            plt.title(f'Feature Importances ({model_name})')
            plt.bar(range(len(indices)), importances[indices])
            plt.xticks(range(len(indices)), [feature_names[i] for i in indices], rotation=45)
            plt.tight_layout()
            plt.show()

    def plot_roc_curves(self, X_train, y_train, X_test, y_test):
        """Построение ROC-кривых для всех моделей"""
        plt.figure(figsize=(10, 8))
        
        for name, model in self.models.items():
            if hasattr(model, "predict_proba"):
                # Train ROC
                y_train_proba = model.predict_proba(X_train)[:, 1]
                fpr_train, tpr_train, _ = roc_curve(y_train, y_train_proba)
                train_auc = auc(fpr_train, tpr_train)
                
                # Test ROC
                y_test_proba = model.predict_proba(X_test)[:, 1]
                fpr_test, tpr_test, _ = roc_curve(y_test, y_test_proba)
                test_auc = auc(fpr_test, tpr_test)
                
                plt.plot(fpr_train, tpr_train, label=f'{name} Train (AUC = {train_auc:.2f})')
                plt.plot(fpr_test, tpr_test, '--', label=f'{name} Test (AUC = {test_auc:.2f})')
        
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC Curves')
        plt.legend()
        plt.show()

    def regularization_analysis(self, X_train, y_train):
        """Анализ влияния регуляризации на модели"""
        alphas = np.logspace(-4, 4, 100)
        
        for reg_name, reg_model in self.regularization_models.items():
            coefs = []
            scores = []
            
            for alpha in alphas:
                reg_model.set_params(alpha=alpha)
                reg_model.fit(X_train, y_train)
                coefs.append(reg_model.coef_)
                scores.append(reg_model.score(X_train, y_train))
            
            self.regularization_results[reg_name] = {
                'alphas': alphas,
                'coefs': np.array(coefs),
                'scores': np.array(scores)
            }
            
        self._plot_regularization_paths()
        
    def _plot_regularization_paths(self):
        """Визуализация путей регуляризации"""
        plt.figure(figsize=(15, 5))
        
        for idx, (name, results) in enumerate(self.regularization_results.items()):
            plt.subplot(1, 3, idx + 1)
            plt.semilogx(results['alphas'], results['coefs'])
            plt.xlabel('Alpha')
            plt.ylabel('Coefficients')
            plt.title(f'{name} Path')
        
        plt.tight_layout()
        plt.show()

class Strategy1:
    def __init__(self, ema_period=20, rsi_period=14, macd_fast=12, macd_slow=26, macd_signal=9):
        self.ema_period = int(ema_period)
        self.rsi_period = int(rsi_period)
        self.macd_fast = int(macd_fast)
        self.macd_slow = int(macd_slow)
        self.macd_signal = int(macd_signal)
        self.ml_models = None
        self.best_params = {}
        self.feature_importances = {}
        self.model_metrics = {}

    def generate_signals(self, data):
        signals = pd.Series(0, index=data.index, dtype=int)
        try:
            df = data.copy()
            
            # Исправлено название колонки на EMA20
            required_columns = ['Close', 'EMA20', 'RSI', 'MACD', 'Signal']
            for col in required_columns:
                if col not in df.columns:
                    df[col] = df['Close'].rolling(20).mean() if col == 'EMA20' else 0.0
                    
            # Обновленные условия с использованием EMA20
            long_mask = (df['Close'] > df['EMA20']) & (df['RSI'] < 65) & (df['MACD'] > df['Signal'])
            short_mask = (df['Close'] < df['EMA20']) & (df['RSI'] > 35) & (df['MACD'] < df['Signal'])
            
            signals.loc[long_mask] = 1
            signals.loc[short_mask] = -1
            
        except Exception as e:
            logging.error(f"Signal generation error: {str(e)}", exc_info=True)
            
        return signals.fillna(0).astype(int)
    
    def optimize_ml_strategy(self, X_train, y_train):
        """Optimize ML models with regularization and visualization"""
        for model_name in ['gradient_boosting', 'random_forest']:
            if model_name == 'gradient_boosting':
                param_grid = {
                    'n_estimators': [100, 200],
                    'learning_rate': [0.01, 0.1],
                    'max_depth': [3, 5],
                    'min_samples_split': [2, 5],
                    'min_samples_leaf': [1, 2],
                    'max_leaf_nodes': [10, 20],
                    'ccp_alpha': [0.0, 0.1]
                }
            else:
                param_grid = {
                    'n_estimators': [100, 200],
                    'max_depth': [3, 5],
                    'min_samples_split': [2, 5],
                    'min_samples_leaf': [1, 2],
                    'max_leaf_nodes': [10, 20],
                    'max_features': ['sqrt', 'log2'],
                    'bootstrap': [True],
                    'ccp_alpha': [0.0, 0.1]
                }

            grid_search = GridSearchCV(
                self.ml_models[model_name],
                param_grid,
                cv=TimeSeriesSplit(n_splits=5),
                scoring='f1',
                n_jobs=-1
            )
            
            grid_search.fit(X_train, y_train)
            self.best_params[model_name] = grid_search.best_params_
            self.ml_models[model_name] = grid_search.best_estimator_
            
            # Store feature importances
            if hasattr(self.ml_models[model_name], 'feature_importances_'):
                self.feature_importances[model_name] = self.ml_models[model_name].feature_importances_

    def optimize_parameters(self, data):
        """Оптимизация параметров стратегии с использованием TimeSeriesSplit."""
        param_grid = {
            'ema_period': range(10, 31, 5),
            'rsi_period': range(10, 21, 2),
            'macd_fast': range(8, 16, 2),
            'macd_slow': range(20, 32, 2),
            'macd_signal': range(7, 12, 1)
        }
        
        best_sharpe = -np.inf
        best_params = {}
        
        # Используем TimeSeriesSplit для кросс-валидации во временных рядах
        tscv = TimeSeriesSplit(n_splits=5)
        
        for params in tqdm(product(*param_grid.values()), 
                          desc="Optimizing Strategy Parameters",
                          total=np.prod([len(v) for v in param_grid.values()])):
            
            param_dict = dict(zip(param_grid.keys(), params))
            strategy = Strategy1(**param_dict)
            
            fold_sharpes = []
            for train_idx, val_idx in tscv.split(data):
                train_data = data.iloc[train_idx]
                val_data = data.iloc[val_idx]
                
                signals = strategy.generate_signals(val_data)
                returns = pd.Series(signals.shift(1) * val_data['Close'].pct_change(), 
                                  index=val_data.index)
                
                if len(returns) > 0:
                    sharpe = np.sqrt(252) * returns.mean() / returns.std() if returns.std() != 0 else 0
                    fold_sharpes.append(sharpe)
            
            avg_sharpe = np.mean(fold_sharpes) if fold_sharpes else -np.inf
            
            if avg_sharpe > best_sharpe:
                best_sharpe = avg_sharpe
                best_params = param_dict
        
        return best_params, best_sharpe

class BacktestEngine:
    def __init__(self, data_handler, model_handler):
        self.data_handler = data_handler
        self.model_handler = model_handler

    def run_backtest(self, strategy, data, train_ratio=0.8):
        try:
            if data.empty:
                raise ValueError("Empty dataset provided")
            
            # Split data into train and valid sets
            train_data, valid_data = self.data_handler.split_data(data, train_ratio)
            result_data = valid_data.copy()
            
            # Run backtest without ML on both train and valid sets
            tech_results = self.run_backtest_without_ml(strategy, data, split_data=True, train_ratio=train_ratio)
            
            # Prepare features for ML models
            features_train = self.data_handler.prepare_ml_features(train_data)
            features_valid = self.data_handler.prepare_ml_features(valid_data)
            
            feature_columns = [
                'EMA20', 'RSI', 'MACD', 'Signal', 'Returns', 'Volatility',
                'MA_Crossover', 'RSI_Change', 'MACD_Hist'
            ] + [f'Returns_Lag{i}' for i in range(1, 6)] + [f'Price_Lag{i}' for i in range(1, 6)]
            
            X_train = features_train[feature_columns].values
            y_train = features_train['Target'].values
            
            X_valid = features_valid[feature_columns].values
            y_valid = features_valid['Target'].values
            
            # Train ML models
            trained_models = self.model_handler.train_models(X_train, y_train)
            strategy.ml_models = trained_models
                        
            # Generate signals and metrics for ML models
            model_results = {}
            for name, model in trained_models.items():
                try:
                    # Train metrics
                    train_pred = model.predict(X_train)
                    train_signals = pd.Series(train_pred, index=features_train.index)
                    train_returns = self._calculate_returns(train_signals, train_data)
                    train_metrics = self._calculate_metrics(train_returns)
                    
                    # Validation metrics
                    valid_pred = model.predict(X_valid)
                    valid_signals = pd.Series(valid_pred, index=features_valid.index)
                    valid_returns = self._calculate_returns(valid_signals, valid_data)
                    valid_metrics = self._calculate_metrics(valid_returns)
                    
                    model_results[name] = {
                        'train_metrics': train_metrics,
                        'valid_metrics': valid_metrics,
                        'signals': valid_signals,
                        'returns': valid_returns
                    }
                except Exception as e:
                    logging.error(f"Error processing {name}: {str(e)}")
                    continue
            
            return {
                'signals': tech_results['signals'],
                'returns': tech_results['returns'],
                'ml_evaluation': self.model_handler.evaluate_models(X_train, y_train, X_valid, y_valid, trained_models),
                'data': result_data,
                'model_results': model_results,
                'without_ml_metrics': {
                    'train': tech_results['train_metrics'],
                    'valid': tech_results['valid_metrics']
                }
            }
            
        except Exception as e:
            logging.error(f"Error in run_backtest: {str(e)}")
            raise
            
    def _calculate_returns(self, signals, data):
        """Safe return calculation with index alignment"""
        try:
            # Align signals with data index
            signals = signals.reindex(data.index, fill_value=0)
            aligned_signals = signals.ffill().bfill().fillna(0)
            position_changes = signals.diff().ffill().fillna(0)
            transaction_costs = abs(position_changes) * 0.001
            price_returns = data['Close'].pct_change()
            
            # Handle potential division by zero in returns calculation
            strategy_returns = pd.Series(index=data.index, dtype=float)
            mask = (signals.shift(1) != 0) & (price_returns != 0)
            strategy_returns[mask] = signals.shift(1)[mask] * price_returns[mask] - transaction_costs[mask]
            strategy_returns = strategy_returns.fillna(0)
            
            return strategy_returns
            
        except Exception as e:
            logging.error(f"Error in _calculate_returns: {str(e)}")
            return pd.Series(0, index=data.index)

    def _calculate_metrics(self, returns):
        """Calculate performance metrics with proper handling"""
        try:
            returns_series = returns if isinstance(returns, pd.Series) else pd.Series(returns)
            
            if len(returns_series) == 0:
                return {}
            
            # Filter valid returns
            positive_returns = returns_series[returns_series > 0]
            negative_returns = returns_series[returns_series < 0]
            
            # Calculate cumulative returns properly
            cum_returns = (1 + returns_series).cumprod()
            rolling_max = cum_returns.expanding().max()
            drawdown = (cum_returns - rolling_max) / rolling_max
            max_drawdown = drawdown.min()
            
            # Calculate annualized metrics
            trading_days = 252
            if len(returns_series) > 0 and cum_returns.iloc[-1] > 0:
                total_return = cum_returns.iloc[-1] - 1
                annualized_return = (1 + total_return) ** (trading_days/len(returns_series)) - 1
            else:
                total_return = -1
                annualized_return = -1
            
            # Calculate ratios safely
            returns_std = returns_series.std()
            negative_returns_std = negative_returns.std()
            
            sharpe_ratio = np.sqrt(trading_days) * returns_series.mean() / returns_std if returns_std != 0 else 0
            sortino_ratio = np.sqrt(trading_days) * returns_series.mean() / negative_returns_std if negative_returns_std != 0 else 0
            
            metrics = {
                'total_return': total_return,
                'annualized_return': annualized_return,
                'sharpe_ratio': sharpe_ratio,
                'sortino_ratio': sortino_ratio,
                'max_drawdown': max_drawdown,
                'win_rate': len(positive_returns) / len(returns_series) if len(returns_series) > 0 else 0,
                'profit_factor': abs(positive_returns.sum() / negative_returns.sum()) if len(negative_returns) > 0 and negative_returns.sum() != 0 else np.inf
            }
            return metrics
            
        except Exception as e:
            logging.error(f"Error in _calculate_metrics: {str(e)}")
            return {}

    def run_backtest_without_ml(self, strategy, data, split_data=True, train_ratio=0.8):
        """Run backtest without ML models with explicit train/valid split"""
        try:
            if data.empty:
                raise ValueError("Empty dataset provided")
    
            if split_data:
                train_data, valid_data = self.data_handler.split_data(data, train_ratio)
                
                # Generate signals and calculate metrics for train set
                train_signals = strategy.generate_signals(train_data)
                train_returns = self._calculate_returns(train_signals, train_data)
                train_metrics = self._calculate_metrics(train_returns)
                
                # Generate signals and calculate metrics for validation set
                valid_signals = strategy.generate_signals(valid_data)
                valid_returns = self._calculate_returns(valid_signals, valid_data)
                valid_metrics = self._calculate_metrics(valid_returns)
                
                return {
                    'train_metrics': train_metrics,
                    'valid_metrics': valid_metrics,
                    'signals': valid_signals,
                    'returns': valid_returns
                }
            else:
                signals = strategy.generate_signals(data)
                returns = self._calculate_returns(signals, data)
                metrics = self._calculate_metrics(returns)
                
                return {
                    'metrics': metrics,
                    'signals': signals,
                    'returns': returns
                }
                
        except Exception as e:
            logging.error(f"Error in run_backtest_without_ml: {str(e)}")
            raise

class Dashboard:
    def __init__(self, results):
        self.results = results
         
    def _create_figure_grid(self, result, symbol):
        """Create enhanced grid of subplots with individual graphs and legends"""
        # Create figure with 2 rows and 4 columns
        fig = make_subplots(
            rows=2, 
            cols=4,
            subplot_titles=(
                # First row - Price charts with signals for each strategy
                f"{symbol} Without ML", 
                f"{symbol} Gradient Boosting",
                f"{symbol} Random Forest",
                f"{symbol} Logistic Regression",
                # Second row - Performance metrics
                "Cumulative Returns",
                "Win Rate",
                "Maximum Drawdown",
                "Daily Returns Distribution"
            ),
            vertical_spacing=0.15,
            horizontal_spacing=0.05,
            specs=[[{}, {}, {}, {}],
                   [{}, {}, {}, {}]]
        )
        
        # Define strategy properties
        strategies = {
            'without_ml': {'name': 'Without ML', 'color': 'gray'},
            'gradient_boosting': {'name': 'Gradient Boosting', 'color': 'green'},
            'random_forest': {'name': 'Random Forest', 'color': 'blue'},
            'logistic_regression': {'name': 'Logistic Regression', 'color': 'red'}
        }
        
        prices = result['with_ml']['data']['Close']
        
        # Plot individual price charts with signals for each strategy
        for col, (strategy_key, props) in enumerate(strategies.items(), 1):
            try:
                # Get signals based on strategy type
                if strategy_key == 'without_ml':
                    signals = result['without_ml']['signals']
                    returns = result['without_ml']['returns']
                else:
                    model_data = result['with_ml']['model_results'].get(strategy_key)
                    signals = model_data['signals'].reindex(prices.index, fill_value=0)
                    returns = model_data['returns'].reindex(prices.index, fill_value=0)
                
                # Add price line
                fig.add_trace(
                    go.Scatter(
                        x=prices.index,
                        y=prices,
                        name="Price",
                        line=dict(color='black', width=1),
                        legendgroup=f'group{col}',
                        showlegend=False
                    ),
                    row=1, col=col
                )
                
                # Add buy signals
                buy_signals = signals[signals > 0]
                fig.add_trace(
                    go.Scatter(
                        x=buy_signals.index,
                        y=prices[buy_signals.index],
                        mode='markers',
                        name='Buy',
                        marker=dict(
                            symbol='triangle-up',
                            color='green',
                            size=8
                        ),
                        legendgroup=f'group{col}',
                        showlegend=True
                    ),
                    row=1, col=col
                )
                
                # Add sell signals
                sell_signals = signals[signals < 0]
                fig.add_trace(
                    go.Scatter(
                        x=sell_signals.index,
                        y=prices[sell_signals.index],
                        mode='markers',
                        name='Sell',
                        marker=dict(
                            symbol='triangle-down',
                            color='red',
                            size=8
                        ),
                        legendgroup=f'group{col}',
                        showlegend=True
                    ),
                    row=1, col=col
                )
                
                # Calculate and store performance metrics for second row
                cum_returns = (1 + returns).cumprod() - 1
                win_rate = (returns > 0).rolling(window=30).mean()
                rolling_max = cum_returns.expanding().max()
                drawdown = (cum_returns - rolling_max) / rolling_max
                
                # Add traces to performance charts (second row)
                # Cumulative Returns
                fig.add_trace(
                    go.Scatter(
                        x=cum_returns.index,
                        y=cum_returns * 100,
                        name=props['name'],
                        line=dict(color=props['color']),
                        showlegend=True
                    ),
                    row=2, col=1
                )
                
                # Win Rate
                fig.add_trace(
                    go.Scatter(
                        x=win_rate.index,
                        y=win_rate * 100,
                        name=props['name'],
                        line=dict(color=props['color']),
                        showlegend=False
                    ),
                    row=2, col=2
                )
                
                # Maximum Drawdown
                fig.add_trace(
                    go.Scatter(
                        x=drawdown.index,
                        y=drawdown * 100,
                        name=props['name'],
                        line=dict(color=props['color']),
                        showlegend=False
                    ),
                    row=2, col=3
                )
                
                # Daily Returns Distribution
                fig.add_trace(
                    go.Histogram(
                        x=returns * 100,
                        name=props['name'],
                        marker_color=props['color'],
                        opacity=0.7,
                        showlegend=False
                    ),
                    row=2, col=4
                )
                
            except Exception as e:
                logging.error(f"Error processing {strategy_key}: {str(e)}")
                continue
        
        # Update layout for better visualization
        fig.update_layout(
            height=1000,
            width=1600,
            showlegend=True,
            margin=dict(t=100, b=50, l=50, r=50),
            template="plotly_white"
        )
        
        # Update axes labels and ranges
        for i in range(1, 5):
            # First row - Price charts
            fig.update_yaxes(title_text="Price", row=1, col=i)
            fig.update_xaxes(title_text="Date", row=1, col=i)
            
        # Second row - Performance metrics
        fig.update_yaxes(title_text="Return (%)", row=2, col=1)
        fig.update_yaxes(title_text="Win Rate (%)", row=2, col=2)
        fig.update_yaxes(title_text="Drawdown (%)", row=2, col=3)
        fig.update_yaxes(title_text="Count", row=2, col=4)
        
        fig.update_xaxes(title_text="Date", row=2, col=1)
        fig.update_xaxes(title_text="Date", row=2, col=2)
        fig.update_xaxes(title_text="Date", row=2, col=3)
        fig.update_xaxes(title_text="Daily Return (%)", row=2, col=4)
        
        return fig
       
    def create_metrics_table(self, result):
        """Create a metrics comparison table with train/valid split including Without ML metrics"""
        metrics_data = {
            'Metric': [
                'Total Return (%)',
                'Annualized Return (%)',
                'Sharpe Ratio',
                'Sortino Ratio',
                'Max Drawdown (%)',
                'Win Rate (%)',
                'Profit Factor'
            ]
        }
        
        # Add Without ML metrics first (now properly extracted and displayed)
        if 'without_ml_metrics' in result:
            # Train metrics
            without_ml_train = result['without_ml_metrics']['train']
            metrics_data['Without ML (Train)'] = [
                f"{without_ml_train.get('total_return', 0)*100:.2f}",
                f"{without_ml_train.get('annualized_return', 0)*100:.2f}",
                f"{without_ml_train.get('sharpe_ratio', 0):.2f}",
                f"{without_ml_train.get('sortino_ratio', 0):.2f}",
                f"{without_ml_train.get('max_drawdown', 0)*100:.2f}",
                f"{without_ml_train.get('win_rate', 0)*100:.2f}",
                f"{without_ml_train.get('profit_factor', 0):.2f}"
            ]
            
            # Valid metrics
            without_ml_valid = result['without_ml_metrics']['valid']
            metrics_data['Without ML (Valid)'] = [
                f"{without_ml_valid.get('total_return', 0)*100:.2f}",
                f"{without_ml_valid.get('annualized_return', 0)*100:.2f}",
                f"{without_ml_valid.get('sharpe_ratio', 0):.2f}",
                f"{without_ml_valid.get('sortino_ratio', 0):.2f}",
                f"{without_ml_valid.get('max_drawdown', 0)*100:.2f}",
                f"{without_ml_valid.get('win_rate', 0)*100:.2f}",
                f"{without_ml_valid.get('profit_factor', 0):.2f}"
            ]
        
        # Add ML model metrics after Without ML metrics
        models = [
            ('gradient_boosting', 'Gradient Boosting'),
            ('random_forest', 'Random Forest'), 
            ('logistic_regression', 'Logistic Regression')
        ]
        
        for model_key, model_name in models:
            if model_key in result['with_ml']['model_results']:
                for prefix, source in [('Train', 'train_metrics'), ('Valid', 'valid_metrics')]:
                    metrics = result['with_ml']['model_results'][model_key][source]
                    col_name = f"{model_name} ({prefix})"
                    
                    metrics_data[col_name] = [
                        f"{metrics.get('total_return', 0) * 100:.2f}",
                        f"{metrics.get('annualized_return', 0) * 100:.2f}",
                        f"{metrics.get('sharpe_ratio', 0):.2f}",
                        f"{metrics.get('sortino_ratio', 0):.2f}",
                        f"{metrics.get('max_drawdown', 0) * 100:.2f}",
                        f"{metrics.get('win_rate', 0) * 100:.2f}",
                        f"{metrics.get('profit_factor', 0):.2f}"
                    ]
        
        # Convert to DataFrame with ordered columns
        df = pd.DataFrame(metrics_data)
        
        # Order columns to ensure Without ML metrics appear first
        ordered_cols = ['Metric']
        if 'Without ML (Train)' in df.columns:
            ordered_cols.extend(['Without ML (Train)', 'Without ML (Valid)'])
        
        # Add remaining ML model columns
        for model_name in ['Gradient Boosting', 'Random Forest', 'Logistic Regression']:
            train_col = f"{model_name} (Train)"
            valid_col = f"{model_name} (Valid)"
            if train_col in df.columns:
                ordered_cols.extend([train_col, valid_col])
        
        return df[ordered_cols].set_index('Metric')
        
    def create_ml_metrics_table(self, result):
        """Create ML metrics table with ROC AUC"""
        ml_metrics = {
            'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC AUC']
        }
        
        if 'with_ml' in result and 'ml_evaluation' in result['with_ml']:
            for model_name, metrics in result['with_ml']['ml_evaluation'].items():
                # Train metrics
                train_metrics = [
                    f"{metrics['train']['accuracy']:.4f}",
                    f"{metrics['train']['precision']:.4f}",
                    f"{metrics['train']['recall']:.4f}",
                    f"{metrics['train']['f1']:.4f}",
                    f"{metrics['train']['roc_auc']:.4f}" if metrics['train']['roc_auc'] is not None else 'N/A'
                ]
                
                # Validation metrics
                valid_metrics = [
                    f"{metrics['valid']['accuracy']:.4f}",
                    f"{metrics['valid']['precision']:.4f}",
                    f"{metrics['valid']['recall']:.4f}",
                    f"{metrics['valid']['f1']:.4f}",
                    f"{metrics['valid']['roc_auc']:.4f}" if metrics['valid']['roc_auc'] is not None else 'N/A'
                ]
                
                ml_metrics[f'{model_name} (Train)'] = train_metrics
                ml_metrics[f'{model_name} (Valid)'] = valid_metrics
        
        # Упорядочиваем колонки согласно требованиям
        ordered_columns = ['Metric']
        model_order = ['gradient_boosting', 'random_forest', 'logistic_regression']
        for model in model_order:
            if f'{model} (Train)' in ml_metrics:
                ordered_columns.extend([f'{model} (Train)', f'{model} (Valid)'])
        
        return pd.DataFrame(ml_metrics)[ordered_columns].set_index('Metric')

    def show_notebook(self):
        """Display the dashboard in Jupyter Notebook"""
        from IPython.display import display, HTML
        
        display(HTML("<h1 style='text-align: center;'>Trading Strategy Analysis Dashboard</h1>"))
        
        # Create dropdown for symbol selection
        symbols = list(self.results.keys())
        dropdown = widgets.Dropdown(
            options=symbols,
            description='Symbol:',
            style={'description_width': 'initial'}
        )
        
        def update_dashboard(symbol):
            if symbol in self.results:
                result = self.results[symbol]
                
                # Display performance metrics
                display(HTML("<h3>Performance Metrics</h3>"))
                metrics_df = self.create_metrics_table(result)
                display(metrics_df.style.background_gradient(cmap='RdYlGn', axis=1))
                
                # Display ML metrics
                display(HTML("<h3>ML Model Metrics</h3>"))
                ml_metrics_df = self.create_ml_metrics_table(result)
                display(ml_metrics_df.style.background_gradient(cmap='RdYlGn', axis=1))

                #display(HTML("<h3>Model ROC AUC Scores</h3>"))
                #display(self.create_ml_metrics_table(result))
                
                # Display charts
                fig = self._create_figure_grid(result, symbol)
                fig.show()
        
        # Connect the dropdown to the update function
        widgets.interact(update_dashboard, symbol=dropdown)
        
    def create_model_comparison_plots(self, result):
        """Создание сравнительных графиков для моделей"""
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('Cumulative Returns', 'Win Rate', 
                          'Maximum Drawdown', 'Daily Returns Distribution')
        )
        
        colors = {'gradient_boosting': 'blue', 
                 'random_forest': 'green',
                 'logistic_regression': 'red'}
        
        for model_name, color in colors.items():
            if model_name in result['with_ml']['model_results']:
                returns = result['with_ml']['model_results'][model_name]['returns']
                
                # Cumulative Returns
                cum_returns = (1 + returns).cumprod()
                fig.add_trace(
                    go.Scatter(x=returns.index, y=cum_returns, 
                              name=f'{model_name} Returns',
                              line=dict(color=color)),
                    row=1, col=1
                )
                
                # Win Rate
                win_rate = (returns > 0).rolling(window=20).mean()
                fig.add_trace(
                    go.Scatter(x=returns.index, y=win_rate,
                              name=f'{model_name} Win Rate',
                              line=dict(color=color)),
                    row=1, col=2
                )
                
                # Drawdown
                peak = cum_returns.expanding(min_periods=1).max()
                drawdown = (cum_returns - peak) / peak
                fig.add_trace(
                    go.Scatter(x=returns.index, y=drawdown,
                              name=f'{model_name} Drawdown',
                              line=dict(color=color)),
                    row=2, col=1
                )
                
                # Returns Distribution
                fig.add_trace(
                    go.Histogram(x=returns, name=f'{model_name} Distribution',
                               marker_color=color, opacity=0.7),
                    row=2, col=2
                )
        
        fig.update_layout(height=800, title_text="Model Comparison")
        return fig

    def create_feature_importance_plot(self, result):
        """Создание графика важности признаков"""
        if 'feature_importances' in result['with_ml']:
            fig = go.Figure()
            
            for model_name, importances in result['with_ml']['feature_importances'].items():
                if isinstance(importances, dict):  # Для моделей с именованными признаками
                    features = list(importances.keys())
                    values = list(importances.values())
                else:  # Для моделей с числовыми индексами признаков
                    features = [f'Feature {i}' for i in range(len(importances))]
                    values = importances
                
                fig.add_trace(
                    go.Bar(name=model_name,
                          x=features,
                          y=values)
                )
            
            fig.update_layout(
                title='Feature Importance by Model',
                xaxis_title='Features',
                yaxis_title='Importance',
                barmode='group'
            )
            
            return fig
        return None
    
def main():
    stocks = ['MSTR']
    cryptos = ['BTC-USD']
    symbols = stocks + cryptos
    start_date = '2020-01-01'

    data_handler = DataHandler()
    model_handler = MLModelHandler()
    backtest_engine = BacktestEngine(data_handler, model_handler)

    results = {}
    
    for symbol in symbols:
        logging.info(f"Processing {symbol}")
        try:
            data_dict = data_handler.download_data([symbol], start_date)
            data = data_dict.get(symbol)

            if data is None or data.empty:
                logging.warning(f"Skipping {symbol} - no data available")
                continue

            if data.shape[0] < 100:
                logging.warning(f"Skipping {symbol} - insufficient data")
                continue

            processed_data = data_handler.preprocess_data(data)
            normalized_data = data_handler.normalize_data(processed_data)

            if normalized_data.empty:
                logging.warning(f"Skipping {symbol} - empty data after normalization")
                continue

            strategy = Strategy1()

            # Run backtest with ML
            result_with_ml = backtest_engine.run_backtest(strategy, normalized_data)

            # Run backtest without ML
            result_without_ml = backtest_engine.run_backtest_without_ml(strategy, normalized_data)
          
            # Save results
            results[symbol] = {
                'with_ml': result_with_ml,
                'without_ml': result_without_ml
            }

            logging.info(f"Completed {symbol}")

        except Exception as e:
            logging.error(f"Error processing {symbol}: {str(e)}")
            continue

    # Display the dashboard in Jupyter Notebook
    dashboard = Dashboard(results)
    dashboard.show_notebook()

if __name__ == "__main__":
    main()

2025-02-28 21:21:09,409 - INFO - Processing MSTR
2025-02-28 21:21:09,410 - INFO - Attempting to download data for MSTR
2025-02-28 21:21:13,563 - ERROR - Error downloading MSTR: Too Many Requests. Rate limited. Try after a while.
2025-02-28 21:21:13,564 - INFO - Attempting alternative download method for MSTR
2025-02-28 21:21:14,984 - ERROR - 
1 Failed download:
2025-02-28 21:21:14,984 - ERROR - ['MSTR']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')
2025-02-28 21:21:16,014 - WARNING - No data was downloaded for any symbol
2025-02-28 21:21:16,015 - WARNING - Skipping MSTR - no data available
2025-02-28 21:21:16,016 - INFO - Processing BTC-USD
2025-02-28 21:21:16,016 - INFO - Attempting to download data for BTC-USD
2025-02-28 21:21:17,456 - ERROR - Error downloading BTC-USD: Too Many Requests. Rate limited. Try after a while.
2025-02-28 21:21:17,457 - INFO - Attempting alternative download method for BTC-USD
2025-02-28 21:21:18,529 - ERROR - 
1 Failed download:


interactive(children=(Dropdown(description='Symbol:', options=(), style=DescriptionStyle(description_width='in…